In [15]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score,train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix


In [2]:
df = pd.read_csv("data/behaviour_data.csv")

In [3]:
df

,timestamp,name,mousePointsCount,avgSpeed,speedStdDev,accelStdDev,jitterCount,angleStd,diffScore,overshootBeforeClick,hoverTime,hesitation,clickLatency,clickOffset,typingStdDev,interActionGapAvg,pageFocusLost,isChecked
0,1767289694905,BotUser,60,0.149797,0.136283,0.070561,6,1.266098,1.696464,0,1368,18,0,43.900409,0,0,False,True
1,1767289727084,Kashish Jaiswal,80,0.835084,2.135536,1.233340,34,2.400265,0.750825,0,0,26937,0,43.741713,0,0,False,True
2,1767291684539,Kashish Jaiswal,85,0.247432,0.311433,0.256825,32,1.722361,1.294357,0,0,1033,0,41.879506,0,0,False,True
3,1767291698229,Kashish Jaiswal,84,0.243080,0.436254,0.411389,35,1.649806,6.507215,0,0,423,0,38.358017,0,0,False,True
4,1767296105859,BotUser,73,0.541225,1.247459,1.483211,10,1.156635,1.779016,0,767,13,0,41.724643,0,0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,1767362569388,kashish,190,0.785889,2.003231,1.496597,48,0.551324,0.436499,0,301,308,994,161.723996,0,0,False,True
87,1767362930634,BotUser,63,0.079527,0.045105,0.035410,2,0.638281,1.801996,0,0,20,0,188.982200,0,0,False,True
88,1767363250533,BotUser,61,0.095876,0.075590,0.099861,2,0.706057,1.253811,0,0,12,0,188.434795,0,0,False,True
89,1767363277766,BotUser,61,0.110147,0.065671,0.038735,0,0.504862,0.265524,0,0,1066,0,187.471790,0,0,False,True


In [4]:
df["label"] = (df["name"]!="BotUser").astype(int)

In [5]:
df.drop(columns = ["timestamp","name","overshootBeforeClick","typingStdDev","interActionGapAvg","pageFocusLost"],inplace = True)

In [6]:
df

,mousePointsCount,avgSpeed,speedStdDev,accelStdDev,jitterCount,angleStd,diffScore,hoverTime,hesitation,clickLatency,clickOffset,isChecked,label
0,60,0.149797,0.136283,0.070561,6,1.266098,1.696464,1368,18,0,43.900409,True,0
1,80,0.835084,2.135536,1.233340,34,2.400265,0.750825,0,26937,0,43.741713,True,1
2,85,0.247432,0.311433,0.256825,32,1.722361,1.294357,0,1033,0,41.879506,True,1
3,84,0.243080,0.436254,0.411389,35,1.649806,6.507215,0,423,0,38.358017,True,1
4,73,0.541225,1.247459,1.483211,10,1.156635,1.779016,767,13,0,41.724643,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,190,0.785889,2.003231,1.496597,48,0.551324,0.436499,301,308,994,161.723996,True,1
87,63,0.079527,0.045105,0.035410,2,0.638281,1.801996,0,20,0,188.982200,True,0
88,61,0.095876,0.075590,0.099861,2,0.706057,1.253811,0,12,0,188.434795,True,0
89,61,0.110147,0.065671,0.038735,0,0.504862,0.265524,0,1066,0,187.471790,True,0


In [9]:
X = df.drop(columns = ["label"])
Y = df["label"]

In [10]:
X

,mousePointsCount,avgSpeed,speedStdDev,accelStdDev,jitterCount,angleStd,diffScore,hoverTime,hesitation,clickLatency,clickOffset,isChecked
0,60,0.149797,0.136283,0.070561,6,1.266098,1.696464,1368,18,0,43.900409,True
1,80,0.835084,2.135536,1.233340,34,2.400265,0.750825,0,26937,0,43.741713,True
2,85,0.247432,0.311433,0.256825,32,1.722361,1.294357,0,1033,0,41.879506,True
3,84,0.243080,0.436254,0.411389,35,1.649806,6.507215,0,423,0,38.358017,True
4,73,0.541225,1.247459,1.483211,10,1.156635,1.779016,767,13,0,41.724643,False
...,...,...,...,...,...,...,...,...,...,...,...,...
86,190,0.785889,2.003231,1.496597,48,0.551324,0.436499,301,308,994,161.723996,True
87,63,0.079527,0.045105,0.035410,2,0.638281,1.801996,0,20,0,188.982200,True
88,61,0.095876,0.075590,0.099861,2,0.706057,1.253811,0,12,0,188.434795,True
89,61,0.110147,0.065671,0.038735,0,0.504862,0.265524,0,1066,0,187.471790,True


In [14]:
X.shape

(91, 12)

In [38]:
Xtrain,Xtest,Ytrain,Ytest = train_test_split(X,Y,test_size=0.25,random_state=42,stratify=Y)

In [39]:
preprocessor = ColumnTransformer(
    transformers=[
        ('isChecked_ohe' , OneHotEncoder(drop ='first',sparse_output= False),["isChecked"])
    ],remainder = 'passthrough'
)

In [40]:
pipe = Pipeline([
    ('preprocessor',preprocessor),
    ('model',RandomForestClassifier(
        n_estimators=12,
        random_state=42
    ))
])

In [41]:
pipe.fit(Xtrain,Ytrain)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('isChecked_ohe', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [42]:
Ypred = pipe.predict(Xtest)

In [43]:
accuracy_score(Ypred,Ytest)

0.9565217391304348

In [44]:
confusion_matrix(Ypred,Ytest)

array([[11,  1],
       [ 0, 11]])

In [51]:
np.mean(cross_val_score(
    pipe,
    X,
    Y,
    cv = 20,
    scoring='accuracy'
))

np.float64(0.96)